# 02 — Clinical EDA: Demographics × Scar Geometry

Goal: characterise the patient population and look for associations between
demographic / clinical variables and MRI-derived scar metrics.

**Sections**
1. Dataset overview (completeness)
2. Demographics (age, sex, time MI→MRI)
3. Risk-factor prevalence
4. Scar geometry distributions
5. Echocardiography
6. Correlation heatmap
7. Subgroup box-plots
8. Outcomes preview

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import scipy.stats as stats

# ── paths ──────────────────────────────────────────────────────────────────
# clinical_data.py lives one directory above this notebook (lv-scar-segmentation/)
MODULE_DIR = Path("..").resolve()
sys.path.insert(0, str(MODULE_DIR))

from clinical_data import load, COLUMN_GROUPS, summary, available_columns

# ▶  CSV registry
CSV_PATH = Path(r"C:\Users\joan\Desktop\FEINA\UPF\TFG\develop-vt.csv")

RESULTS_DIR = MODULE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# ── style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})
CMAP = "Blues"
PAL  = plt.cm.tab10.colors

## 1 · Dataset overview

In [ ]:
df = load(CSV_PATH)
print(f"Rows: {len(df)}   Columns: {len(df.columns)}")
summary(df)

In [ ]:
# ── Completeness heat-map ──────────────────────────────────────────────────
grp = available_columns(df)
rows = []
for group, cols in grp.items():
    for c in cols:
        rows.append({"group": group, "column": c,
                     "pct_non_null": df[c].notna().mean() * 100})
comp = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(14, 0.35 * len(comp) + 1))
pivot = comp.pivot(index="column", columns="group", values="pct_non_null")
# reorder rows by group
col_order = list(grp.keys())
ordered = []
for g in col_order:
    ordered += [c for c in grp[g] if c in pivot.index]
pivot = pivot.loc[ordered]

im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=100)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=35, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
plt.colorbar(im, ax=ax, label="% non-null", shrink=0.5)
ax.set_title("Completeness by column group")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_completeness.png", bbox_inches="tight")
plt.show()

## 2 · Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# -- Age distribution -------------------------------------------------------
ax = axes[0]
age = df["age"].dropna()
ax.hist(age, bins=15, color=PAL[0], edgecolor="white", linewidth=0.5)
ax.axvline(age.mean(), color="red", linestyle="--", label=f"mean {age.mean():.1f}")
ax.axvline(age.median(), color="orange", linestyle=":", label=f"median {age.median():.1f}")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Patients")
ax.set_title("Age distribution")
ax.legend(fontsize=8)

# -- Sex breakdown ----------------------------------------------------------
ax = axes[1]
sex_counts = df["sex"].value_counts(dropna=False).sort_index()
labels = {0: "Female", 1: "Male"}
sex_labels = [labels.get(k, f"Unknown ({k})") for k in sex_counts.index]
ax.bar(sex_labels, sex_counts.values, color=[PAL[3], PAL[0]])
for i, v in enumerate(sex_counts.values):
    ax.text(i, v + 0.3, str(v), ha="center", fontsize=9)
ax.set_ylabel("Patients")
ax.set_title("Sex distribution")

# -- Time MI → MRI ----------------------------------------------------------
ax = axes[2]
if {"date_mi", "date_mri"}.issubset(df.columns):
    delay = (df["date_mri"] - df["date_mi"]).dt.days.dropna()
    delay_m = delay / 30.44   # convert to months
    ax.hist(delay_m, bins=20, color=PAL[2], edgecolor="white", linewidth=0.5)
    ax.axvline(delay_m.median(), color="orange", linestyle=":",
               label=f"median {delay_m.median():.0f} mo")
    ax.set_xlabel("Months from MI to MRI")
    ax.set_ylabel("Patients")
    ax.set_title("MI → MRI delay")
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, "date_mi or date_mri missing",
            ha="center", va="center", transform=ax.transAxes)

plt.suptitle("Demographics", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_demographics.png", bbox_inches="tight")
plt.show()

## 3 · Risk-factor prevalence

In [ ]:
rf_cols  = [c for c in ["hta", "dlp", "dm", "smoking", "afib", "mi_yn"] if c in df.columns]
rf_pct   = df[rf_cols].apply(lambda s: (s == 1).sum() / s.notna().sum() * 100)
rf_count = df[rf_cols].apply(lambda s: (s == 1).sum())

labels_map = {"hta": "Hypertension", "dlp": "Dyslipidaemia", "dm": "Diabetes",
              "smoking": "Smoking", "afib": "AF", "mi_yn": "Prior MI"}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh([labels_map.get(c, c) for c in rf_pct.index],
               rf_pct.values, color=PAL[0])
for bar, n in zip(bars, rf_count.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"n={n}", va="center", fontsize=9)
ax.set_xlabel("Prevalence (%)")
ax.set_xlim(0, 110)
ax.set_title("Risk factor prevalence")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_risk_factors.png", bbox_inches="tight")
plt.show()

# NYHA
if "nyha" in df.columns:
    print("\nNYHA class distribution:")
    print(df["nyha"].value_counts(dropna=False).sort_index())

## 4 · Scar geometry

In [ ]:
# ── Continuous scar metrics ────────────────────────────────────────────────
scar_num = [c for c in ["lv_mass_g", "core_g", "core_pct",
                         "bz_g", "bz_pct", "bz_core_g", "bz_core_pct",
                         "channel_mass_g"] if c in df.columns]

n_cols = 4
n_rows = int(np.ceil(len(scar_num) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(scar_num):
    s = df[col].dropna()
    axes[i].hist(s, bins=20, color=PAL[1], edgecolor="white", linewidth=0.5)
    axes[i].axvline(s.median(), color="orange", linestyle=":",
                    label=f"med {s.median():.1f}")
    axes[i].set_title(col)
    axes[i].set_xlabel("value")
    axes[i].legend(fontsize=7)

for ax in axes[len(scar_num):]:
    ax.set_visible(False)

plt.suptitle("Scar geometry — continuous metrics", fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_scar_distributions.png", bbox_inches="tight")
plt.show()

print("\nDescriptive stats:")
display(df[scar_num].describe().round(2))

In [ ]:
# ── Categorical scar metrics ───────────────────────────────────────────────
cat_cols = [c for c in ["enhancement_territory", "enhancement_distribution",
                         "enhancement_grade", "channels", "mri_type"] if c in df.columns]

fig, axes = plt.subplots(1, len(cat_cols), figsize=(4 * len(cat_cols), 4))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    vc = df[col].value_counts(dropna=False).head(10)
    ax.barh(vc.index.astype(str), vc.values, color=PAL[4])
    ax.set_title(col)
    ax.set_xlabel("n")
    ax.tick_params(axis="y", labelsize=7)

plt.suptitle("Scar geometry — categorical", fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_scar_categorical.png", bbox_inches="tight")
plt.show()

## 5 · Echocardiography

In [ ]:
echo_num = [c for c in COLUMN_GROUPS["echo"] if c in df.columns]

n_cols = 5
n_rows = int(np.ceil(len(echo_num) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(echo_num):
    s = df[col].dropna()
    axes[i].hist(s, bins=18, color=PAL[6], edgecolor="white", linewidth=0.5)
    axes[i].axvline(s.median(), color="orange", linestyle=":",
                    label=f"med {s.median():.1f}")
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

for ax in axes[len(echo_num):]:
    ax.set_visible(False)

plt.suptitle("Echocardiography", fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_echo.png", bbox_inches="tight")
plt.show()

# LVEF vs scar burden scatter
if {"lvef_pct", "core_pct"}.issubset(df.columns):
    sub = df[["lvef_pct", "core_pct"]].dropna()
    r, p = stats.pearsonr(sub["lvef_pct"], sub["core_pct"])
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(sub["lvef_pct"], sub["core_pct"], alpha=0.6, s=30, color=PAL[0])
    m, b = np.polyfit(sub["lvef_pct"], sub["core_pct"], 1)
    xs = np.linspace(sub["lvef_pct"].min(), sub["lvef_pct"].max(), 100)
    ax.plot(xs, m * xs + b, color="red", linewidth=1.2,
            label=f"r={r:.2f}  p={p:.3f}")
    ax.set_xlabel("LVEF (%)")
    ax.set_ylabel("Core scar (%)")
    ax.set_title("LVEF vs Core scar burden")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "eda_lvef_vs_core.png", bbox_inches="tight")
    plt.show()

## 6 · Correlation: demographics + risk factors × scar geometry

In [ ]:
demo_rf = [c for c in
           ["age", "sex", "hta", "dlp", "dm", "smoking", "afib", "nyha",
            "lvef_pct", "lvedv_ml", "lvesv_ml"]
           if c in df.columns]

scar_vars = [c for c in
             ["lv_mass_g", "core_g", "core_pct", "bz_g", "bz_pct",
              "bz_core_g", "bz_core_pct", "channel_mass_g", "channels",
              "enhancement_grade"]
             if c in df.columns]

corr_df = df[demo_rf + scar_vars].apply(pd.to_numeric, errors="coerce")
corr = corr_df.corr(method="pearson")

# Show only the cross-block (demo/rf rows × scar cols)
cross = corr.loc[demo_rf, scar_vars]

fig, ax = plt.subplots(figsize=(len(scar_vars) * 1.1 + 1,
                                len(demo_rf) * 0.55 + 1))
im = ax.imshow(cross.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(scar_vars)))
ax.set_xticklabels(scar_vars, rotation=40, ha="right", fontsize=9)
ax.set_yticks(range(len(demo_rf)))
ax.set_yticklabels(demo_rf, fontsize=9)
plt.colorbar(im, ax=ax, label="Pearson r", shrink=0.6)

# annotate cells
for i in range(len(demo_rf)):
    for j in range(len(scar_vars)):
        v = cross.iloc[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=7, color="black" if abs(v) < 0.5 else "white")

ax.set_title("Correlation: clinical variables × scar geometry",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_correlation_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Top correlations table ─────────────────────────────────────────────────
cross_flat = (
    cross.stack()
    .reset_index()
    .rename(columns={"level_0": "clinical", "level_1": "scar", 0: "r"})
)
cross_flat["abs_r"] = cross_flat["r"].abs()
print("Top 15 correlations (|r| largest):")
display(
    cross_flat.sort_values("abs_r", ascending=False)
    .head(15)
    .drop(columns="abs_r")
    .reset_index(drop=True)
)

## 7 · Subgroup box-plots

In [ ]:
def boxplot_by_group(df, group_col, value_col, labels=None, title=None, ax=None):
    """Box-plot of value_col split by binary group_col, with Mann-Whitney p."""
    sub = df[[group_col, value_col]].dropna()
    groups = sorted(sub[group_col].unique())
    data   = [sub.loc[sub[group_col] == g, value_col].values for g in groups]
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    bp = ax.boxplot(data, patch_artist=True, notch=False,
                    medianprops=dict(color="black", linewidth=1.5))
    for patch, color in zip(bp["boxes"], PAL):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    tick_labels = [labels.get(g, str(g)) if labels else str(g) for g in groups]
    ax.set_xticklabels(tick_labels)
    ax.set_ylabel(value_col)
    ax.set_title(title or f"{value_col} by {group_col}")
    # Mann-Whitney U
    if len(data) == 2 and len(data[0]) > 0 and len(data[1]) > 0:
        u, p = stats.mannwhitneyu(data[0], data[1], alternative="two-sided")
        ax.text(0.97, 0.97, f"p={p:.3f}", transform=ax.transAxes,
                ha="right", va="top", fontsize=8,
                color="green" if p < 0.05 else "grey")
    return ax


# ── 2×4 grid: key scar metrics by sex, HTA, DM, AFIB ─────────────────────
scar_targets = [c for c in ["core_pct", "bz_core_pct"] if c in df.columns]
group_vars   = [c for c in ["sex", "hta", "dm", "afib"] if c in df.columns]
group_labels = {"sex": {0: "Female", 1: "Male"},
                "hta": {0: "No HTA", 1: "HTA"},
                "dm":  {0: "No DM", 1: "DM"},
                "afib": {0: "No AF", 1: "AF"}}

fig, axes = plt.subplots(len(scar_targets), len(group_vars),
                          figsize=(4 * len(group_vars), 4 * len(scar_targets)))
if len(scar_targets) == 1:
    axes = [axes]

for row, scar in enumerate(scar_targets):
    for col, grp in enumerate(group_vars):
        boxplot_by_group(df, grp, scar,
                         labels=group_labels.get(grp, {}),
                         ax=axes[row][col])

plt.suptitle("Scar burden by clinical subgroups", fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_subgroup_boxplots.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Scar by enhancement territory (top 6 territories) ─────────────────────
if {"enhancement_territory", "core_pct"}.issubset(df.columns):
    top_terr = df["enhancement_territory"].value_counts().head(6).index
    sub = df[df["enhancement_territory"].isin(top_terr)][["enhancement_territory", "core_pct"]].dropna()

    order = sub.groupby("enhancement_territory")["core_pct"].median().sort_values(ascending=False).index
    data  = [sub.loc[sub["enhancement_territory"] == t, "core_pct"].values for t in order]

    fig, ax = plt.subplots(figsize=(10, 4))
    bp = ax.boxplot(data, patch_artist=True,
                    medianprops=dict(color="black", linewidth=1.5))
    for patch, color in zip(bp["boxes"], PAL):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels(order, rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("Core scar (%)")
    ax.set_title("Core scar burden by enhancement territory")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "eda_scar_by_territory.png", bbox_inches="tight")
    plt.show()

In [ ]:
# ── Age vs scar burden scatter (coloured by sex) ───────────────────────────
if {"age", "core_pct", "sex"}.issubset(df.columns):
    sub = df[["age", "core_pct", "sex"]].dropna()
    fig, ax = plt.subplots(figsize=(6, 5))
    for sex_val, label, color in [(0, "Female", PAL[3]), (1, "Male", PAL[0])]:
        g = sub[sub["sex"] == sex_val]
        ax.scatter(g["age"], g["core_pct"], alpha=0.65, s=35,
                   color=color, label=label)
    # overall regression line
    m, b = np.polyfit(sub["age"], sub["core_pct"], 1)
    r, p = stats.pearsonr(sub["age"], sub["core_pct"])
    xs = np.linspace(sub["age"].min(), sub["age"].max(), 100)
    ax.plot(xs, m * xs + b, "k--", linewidth=1.2, label=f"r={r:.2f} p={p:.3f}")
    ax.set_xlabel("Age (years)")
    ax.set_ylabel("Core scar (%)")
    ax.set_title("Age vs Core scar burden")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "eda_age_vs_core.png", bbox_inches="tight")
    plt.show()

## 8 · Outcomes preview

In [ ]:
# ── Event rates ───────────────────────────────────────────────────────────
outcome_cols = [c for c in ["death_yn", "cardiac_death_yn", "arrhythmic_death_yn",
                              "followup_va_yn", "admission_hf", "admission_angina"]
                if c in df.columns]

if outcome_cols:
    rates = pd.DataFrame({
        "event": outcome_cols,
        "n": [int((df[c] == 1).sum()) for c in outcome_cols],
        "pct": [(df[c] == 1).sum() / df[c].notna().sum() * 100 for c in outcome_cols],
    })
    print(rates.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(rates["event"], rates["pct"], color=PAL[7])
    for i, (n, pct) in enumerate(zip(rates["n"], rates["pct"])):
        ax.text(pct + 0.5, i, f"n={n}", va="center", fontsize=9)
    ax.set_xlabel("Prevalence (%)")
    ax.set_xlim(0, 100)
    ax.set_title("Clinical outcomes")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "eda_outcomes.png", bbox_inches="tight")
    plt.show()

In [ ]:
# ── Scar burden: event vs no-event (for each outcome) ─────────────────────
scar_metric = "core_pct" if "core_pct" in df.columns else None

if scar_metric and outcome_cols:
    fig, axes = plt.subplots(1, len(outcome_cols),
                              figsize=(3.5 * len(outcome_cols), 4))
    if len(outcome_cols) == 1:
        axes = [axes]

    for ax, oc in zip(axes, outcome_cols):
        boxplot_by_group(df, oc, scar_metric,
                         labels={0: "No", 1: "Yes"},
                         title=f"{scar_metric}\nby {oc}",
                         ax=ax)

    plt.suptitle("Core scar burden by clinical outcome", fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "eda_scar_by_outcome.png", bbox_inches="tight")
    plt.show()